# Python Toolkit Guide: VS Code, data libraries, uv and GitHub

A practical reference for setting up a modern Python environment and building projects on your own: the editor, the libraries, the package tools, and GitHub.

Run the cells from top to bottom (**Run All**). The demos use simulated data, so no API keys
or database are needed.

**Packages used:** numpy, pandas, polars, pyarrow, matplotlib, plotly, scikit-learn.
Install them in your project with `uv add numpy pandas polars pyarrow matplotlib plotly scikit-learn ipykernel`.
DuckDB is optional (`uv add duckdb`).

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd

# Which Python is this notebook using? It should point inside your project's .venv
print(sys.executable)

## 1. VS Code

VS Code is a free editor that becomes a full Python and data environment once you add a few extensions. It handles scripts, Jupyter notebooks, Git, the terminal and remote servers in one window.

### Extensions to install first
| Extension | What it adds |
| --- | --- |
| Python (Microsoft) | Run, debug and pick the Python interpreter |
| Pylance | Autocomplete, type checking, go to definition |
| Jupyter | Run .ipynb notebooks inside VS Code |
| Ruff | Fast linting and auto-formatting on save |
| GitLens | Shows who changed each line and when; richer Git history |
| Remote - SSH | Edit and run code on another machine, such as a Linux server, as if it were local |
| Rainbow CSV / Data Wrangler | Readable CSV files; a spreadsheet-like view of DataFrames |
| Claude Code (optional) | An AI assistant that can read, explain and edit code in your project |

### Features worth learning early
- **Command Palette.** Cmd+Shift+P on Mac, Ctrl+Shift+P on Windows. Every command in VS Code is searchable here.
- **Select Interpreter.** Tells VS Code which Python (and which virtual environment) to use. Most 'module not found' problems come from this.
- **Integrated terminal.** Ctrl+` opens a terminal in your project folder.
- **Debugger.** Click left of a line number to set a breakpoint, press F5, and step through the code while inspecting variables.
- **Source Control panel.** Stage, commit, push and view diffs without typing Git commands.
- **Notebook cells in .py files.** Typing # %% in a regular Python file creates runnable cells: notebook-style exploration in a plain script that Git handles well.

### Handy shortcuts
| Action | Mac | Windows |
| --- | --- | --- |
| Open any file by name | Cmd+P | Ctrl+P |
| Command Palette | Cmd+Shift+P | Ctrl+Shift+P |
| Go to definition | F12 | F12 |
| Rename a variable everywhere | F2 | F2 |
| Comment out lines | Cmd+/ | Ctrl+/ |
| Multi-cursor on next match | Cmd+D | Ctrl+D |
| Toggle terminal | Ctrl+` | Ctrl+` |

### Gotchas
- **Wrong interpreter.** A package is installed but VS Code says it is missing: the bottom-right corner shows which Python is active. Pick the one inside your project's .venv.
- **Notebook kernel vs terminal.** A notebook's kernel and the terminal can use different environments. Check both.
- **Open the folder, not the file.** Use File > Open Folder on the project root so VS Code finds the .venv, settings and Git repo.
- **Workspace settings.** Settings in .vscode/settings.json apply only to that project and override your global ones.

The cell above printed `sys.executable`: the quickest way to check the "wrong interpreter" gotcha.

## 2. Data analysis libraries

NumPy and pandas are the foundation almost everything else builds on. The biggest recent trends are faster tools for the same jobs: Polars and DuckDB for data, uv and Ruff for tooling, marimo for notebooks, and PyTorch replacing TensorFlow for most new deep learning work.

### Core data
| Library | Used for | Gotcha |
| --- | --- | --- |
| NumPy | Fast arrays and math; the base layer under pandas, scikit-learn and others | Never compare floats with ==; use np.isclose() |
| pandas | Tables (DataFrames): load, clean, filter, group, join, time series | Chained assignment like df[df.x > 0]['y'] = 1 does not change df; use df.loc[df.x > 0, 'y'] = 1 |
| Polars (trending) | A much faster DataFrame library, often 5 to 10 times quicker on large data | Different syntax from pandas (expressions, no index); lazy queries need .collect() |
| DuckDB (trending) | Run SQL directly on DataFrames, CSV and Parquet files, with no server | Only one process can write to a DuckDB file at a time |
| PyArrow / Parquet | A compact, typed file format; much faster than CSV for large datasets | CSV loses types (dates become text); save working data as Parquet |

### Visualization
| Library | Used for | Gotcha |
| --- | --- | --- |
| Matplotlib | The standard static plotting library; full control | Two styles exist (plt.plot vs fig, ax); use fig, ax = plt.subplots() consistently |
| Seaborn | Statistical charts (distributions, heatmaps) in one line | Works best with 'long' data: one row per observation |
| Plotly | Interactive charts with zoom and hover, for notebooks and web apps | Very large datasets make slow, heavy pages; downsample first |

### Stats and ML
| Library | Used for | Gotcha |
| --- | --- | --- |
| SciPy | Statistical tests, optimization, signal processing | Check each test's assumptions (normality, independence) |
| statsmodels | Regression with full statistical output, time-series models (ARIMA) | sm.OLS needs an explicit constant (sm.add_constant) unless you use the formula API |
| scikit-learn | Classic ML: regression, classification, clustering, PCA, pipelines | Data leakage: fit scalers and models on training data only; for time series use TimeSeriesSplit, never shuffled splits |
| XGBoost / LightGBM | Gradient boosting; usually the strongest choice for tabular data | Easy to overfit; tune with cross-validation |
| PyTorch (trending) | Deep learning; now the most common choice for new work | Often overkill for tabular data; try gradient boosting first |

### Notebooks and apps
| Library | Used for | Gotcha |
| --- | --- | --- |
| Jupyter | Interactive notebooks for exploration | Cells run out of order hide bugs; use Restart and Run All before trusting results |
| marimo (trending) | Reactive notebooks saved as plain .py files, so they work well with Git | Each variable can only be defined in one cell |
| Streamlit | Turn a script into an interactive web dashboard | The whole script reruns on every click; cache slow loads with @st.cache_data |

### Getting data
| Library | Used for | Gotcha |
| --- | --- | --- |
| requests / httpx | Calling web APIs | No default timeout; always pass timeout=10 or a call can hang forever |
| SQLAlchemy | Connecting to MySQL, Postgres and other databases | Use parameters (:name), never f-strings, to put values into SQL |
| BeautifulSoup / Playwright | Web scraping: static HTML / pages that need a real browser | Check the site's terms and robots.txt; add delays between requests |
| Pydantic | Validating data shapes and types (API responses, configs) | Version 2 changed much of the API; older tutorials use version 1 |

### Gotcha demos

**NumPy: never compare floats with `==`**

In [ ]:
print(0.1 + 0.2 == 0.3)            # False: floats are stored in binary, not exactly
print(np.isclose(0.1 + 0.2, 0.3))  # True: compares within a small tolerance

**pandas: chained assignment does not change the DataFrame**

In [ ]:
df = pd.DataFrame({"x": [-1, 2, 3], "y": [0, 0, 0]})

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    df[df.x > 0]["y"] = 1          # writes to a temporary copy
print("chained:\n", df)

df.loc[df.x > 0, "y"] = 1          # the right way: one .loc call
print("\nwith .loc:\n", df)

**Polars: same job, different syntax, and lazy queries need `.collect()`**

In [ ]:
import polars as pl

pdf = pl.DataFrame({"symbol": ["AAPL", "AAPL", "NVDA"], "price": [180.0, 182.5, 120.0]})

# Eager: runs immediately
print(pdf.group_by("symbol").agg(pl.col("price").mean()))

# Lazy: builds a plan, nothing runs until .collect()
plan = pdf.lazy().filter(pl.col("price") > 150)
print(type(plan))
print(plan.collect())

**DuckDB: SQL directly on a pandas DataFrame** (skipped if DuckDB is not installed)

In [ ]:
try:
    import duckdb
    quotes = pd.DataFrame({"symbol": ["AAPL", "NVDA", "AAPL"], "price": [180, 120, 182]})
    print(duckdb.sql("SELECT symbol, AVG(price) AS avg_price FROM quotes GROUP BY symbol").df())
except ImportError:
    print("DuckDB not installed. Install it with: uv add duckdb")

**CSV loses types, Parquet keeps them**

In [ ]:
df = pd.DataFrame({"ts": pd.date_range("2026-01-01", periods=3), "price": [1.0, 2.0, 3.0]})
df.to_csv("demo.csv", index=False)
df.to_parquet("demo.parquet")

print("from CSV:    ", pd.read_csv("demo.csv").dtypes["ts"])          # object (text)
print("from Parquet:", pd.read_parquet("demo.parquet").dtypes["ts"])  # datetime64

**scikit-learn: for time series, never shuffle the split**

In [ ]:
from sklearn.model_selection import KFold, TimeSeriesSplit

days = np.arange(10)
shuffled = next(KFold(n_splits=5, shuffle=True, random_state=0).split(days))
ordered = next(TimeSeriesSplit(n_splits=4).split(days))

print("Shuffled  train:", shuffled[0], "test:", shuffled[1])  # trains on the future
print("TimeSeries train:", ordered[0], "test:", ordered[1])   # only the past predicts the future

**Matplotlib: use the `fig, ax` style consistently**

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([1, 2, 3, 4], [10, 12, 9, 14])
ax.set_title("fig, ax style")
ax.set_xlabel("Day")
ax.set_ylabel("Price")
plt.show()

## 3. How they work together

A typical analysis moves through the same stages: get data, clean it, explore it, model it,
show it. Here step 1 simulates prices instead of reading them from MySQL; the commented line
shows the real version.

In [ ]:
import plotly.express as px
from sklearn.linear_model import LinearRegression

# 1. Get data (simulated). With a database it would be:
# df = pd.read_sql("SELECT symbol, ts, price FROM quotes", engine, parse_dates=["ts"])
rng = np.random.default_rng(7)
dates = pd.bdate_range("2025-01-01", periods=250)
df = pd.concat([
    pd.DataFrame({"symbol": s, "ts": dates, "price": p0 * np.exp(np.cumsum(rng.normal(0.0005, 0.018, 250)))})
    for s, p0 in {"AAPL": 180, "NVDA": 120, "MSFT": 400}.items()
], ignore_index=True)

# 2. Clean and reshape with pandas
df = df.dropna().sort_values("ts")
df["return"] = df.groupby("symbol")["price"].pct_change()

# 3. Explore: summary stats and an interactive chart
display(df.groupby("symbol")["return"].describe().round(4))
px.line(df, x="ts", y="price", color="symbol", title="Simulated prices").show()

# 4. Model with scikit-learn (NumPy arrays underneath)
aapl = df[df.symbol == "AAPL"].dropna()
X = np.arange(len(aapl)).reshape(-1, 1)
model = LinearRegression().fit(X, aapl["price"])
print("AAPL trend per trading day:", round(model.coef_[0], 3))

# 5. Save for later in a fast, typed format
df.to_parquet("prices.parquet")

## 4. venv vs uv

Both give each project its own private set of packages, so one project's versions never break another's. venv is built into Python and only creates the environment; uv is a newer, much faster tool that also installs Python itself, manages dependencies and locks exact versions. For new projects, uv is the recommended choice; venv is still worth knowing because older tutorials and projects use it.

|  | venv + pip | uv |
| --- | --- | --- |
| Comes with Python | Yes | No, one-time install |
| Speed | Normal | Often 10 to 100 times faster installs |
| Installs Python versions | No | Yes (uv python install 3.13) |
| Dependency list | requirements.txt, maintained by hand | pyproject.toml, updated automatically |
| Exact versions locked | Only if you run pip freeze | Always, in uv.lock |
| Need to activate the environment | Yes | No; uv run uses it automatically |

### The same tasks, side by side
| Task | venv + pip | uv |
| --- | --- | --- |
| Start a project | `python -m venv .venv` | `uv init` |
| Activate (Mac/Linux) | `source .venv/bin/activate` | `# not needed` |
| Add a package | `pip install pandas` | `uv add pandas` |
| Save the dependency list | `pip freeze > requirements.txt` | `# automatic` |
| Recreate on another machine | `pip install -r requirements.txt` | `uv sync` |
| Run a script | `python etl.py   # after activating` | `uv run etl.py` |

Install uv on a Mac or Linux:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

### Gotchas
- **Never commit .venv.** Add it to .gitignore. Commit pyproject.toml and uv.lock (or requirements.txt) instead.
- **Forgetting to activate (venv only).** pip install then installs into the global Python, not the project.
- **Mixing tools.** In a uv project use uv add, not pip install, or the lock file falls out of date.
- **Conda.** A third option, mainly used for heavy scientific packages with non-Python dependencies; for most data work uv is simpler.

In [ ]:
# Is this notebook running inside a virtual environment?
print("Environment:", sys.prefix)
print("Inside a venv:", sys.prefix != sys.base_prefix)

## 5. Git and GitHub

Git is version control that runs on your computer: it saves snapshots (commits) of your project so you can see every change and go back to any point. GitHub is a website that hosts Git repositories online, so you can back them up, share them and work with other people.

![Git workflow](assets/git_workflow.png)

Nothing reaches GitHub until you push, and nothing from GitHub reaches your computer until you pull.

### Core ideas
| Term | Meaning |
| --- | --- |
| Repository (repo) | A project folder whose history Git tracks |
| Commit | A saved snapshot with a message describing the change |
| Branch | A separate line of work; main is the default, and new ideas go on their own branch |
| Remote | The online copy of the repo, usually called origin |
| Clone | Download a repo from GitHub to your computer |
| Fork | Your own copy of someone else's repo on GitHub, which you can change freely |
| Pull request (PR) | A request to merge a branch into another, with a place for review and discussion |
| Issue | A bug report, question or idea attached to a repo |
| .gitignore | A list of files Git should never track, such as .venv and .env |

### The daily workflow
```bash
git clone https://github.com/user/project.git   # once, to get a copy
git switch -c add-news-tool                      # new branch for a new idea
# ...edit files...
git status                                       # what changed?
git add etl.py                                   # stage the changes to save
git commit -m "Add news to the ETL pipeline"     # save a snapshot
git push -u origin add-news-tool                 # upload the branch to GitHub
# then open a pull request on GitHub and merge it into main
git switch main && git pull                      # get the merged result
```

### Gotchas
- **Secrets in commits.** An API key committed once stays in the history even after you delete the file. Keep keys in .env, list .env in .gitignore, and if a key is ever pushed, revoke it and make a new one.
- **Large data files.** GitHub rejects files over 100 MB and repos get slow well before that. Keep data out of Git; commit the code that fetches it.
- **Merge conflicts.** When two changes touch the same lines, Git asks you to choose. VS Code shows both versions side by side with Accept buttons.
- **Pull before you push.** If GitHub has changes you do not have, your push is rejected; run git pull first.
- **Small, frequent commits.** Clear messages make the history useful; one giant 'updates' commit tells you nothing later.

## 6. Getting the most out of GitHub

GitHub is also the best free classroom for programming: millions of real projects are open
to read, run, question and improve.

### For learning
- **Read real code.** Pick a library you use (pandas, Streamlit, FastMCP) and browse its source. Press . on any repo page to open it in a browser version of VS Code.
- **Star and watch.** Starring bookmarks a repo; watching (Releases only) tells you when a new version ships.
- **Explore and Trending pages.** See what developers are building right now, filtered by language.
- **Awesome lists.** Search for lists such as awesome-quant or awesome-python: curated directories of the best tools on a topic.
- **Examples and Issues.** Check a repo's examples folder and its Issues. Closed issues often answer the exact problem you hit.
- **Codespaces.** Run any repo in the browser without installing anything.

### For your own projects
- **A repo for every project.** From day one, even private ones. It is a backup, a history and a portfolio.
- **A good README.** What the project does, how to install it, how to run it, and a screenshot.
- **Issues as a to-do list.** Plus a Project board to organize them.
- **GitHub Actions.** Runs code automatically on a schedule or on every push: tests, linting, or even a daily data download.
- **Gists.** Handy for sharing a single snippet or notebook.

### Working with others
- **Start small.** Look for issues labelled good first issue or help wanted. Typo and documentation fixes are welcome first contributions.
- **Read CONTRIBUTING.md.** Before opening a pull request; it explains the project's rules.
- **The usual contribution flow.** Fork the repo, clone your fork, create a branch, commit, push, and open a pull request back to the original.
- **Write useful issues.** What you expected, what happened, the exact error, and the smallest code that reproduces it.
- **Use Discussions.** For questions and ideas rather than Issues, when a repo has them enabled.
- **Review others' pull requests.** Reading and commenting on PRs is one of the fastest ways to learn how experienced developers think.
- **Keep PRs small and focused.** One change per PR gets reviewed and merged much faster.

## 7. A suggested order

Each step uses the one before it, and the projects give each skill a real job to do.

1. **VS Code and uv.** Install both, create a first project with uv init, and run a script from VS Code.
2. **Git and GitHub.** Turn that project into a repo, add a .gitignore, and push it to GitHub.
3. **pandas in a notebook.** Load a CSV or an API response into a DataFrame and explore it with Plotly.
4. **Database connectivity.** Read from and write to MySQL with SQLAlchemy.
5. **The ETL project.** Combine the first four steps into a script that runs on a schedule.
6. **Stats and ML.** Add statsmodels or scikit-learn to the collected data.
7. **Share it.** A Streamlit dashboard or an MCP server, with a README on GitHub.

In [ ]:
# Clean up the demo files this notebook created
from pathlib import Path
for f in ["demo.csv", "demo.parquet", "prices.parquet"]:
    Path(f).unlink(missing_ok=True)